## SUPPORT VECTOR MACHINE

```
Dataset Selection:
Data set Description described in another document “Drug Response Classification”

Task 1: Exploratory Data Analysis (EDA)
1.	Load the dataset and perform fundamental data exploration.
2.	Utilize histograms, box plots, or density plots to understand feature distributions.
3.	Investigate feature correlations to discern relationships within the data.

Task 2: Data Preprocessing
1.	Encode categorical variables if necessary.
2.	Split the dataset into training and testing sets.

Task 3: Data Visualization
1.	Employ scatter plots, pair plots, or relevant visualizations to comprehend feature distributions and relationships.
2.	Visualize class distributions to gauge dataset balance or imbalance.

Task 4: SVM Implementation
1.	Implement a basic SVM classifier using Python libraries like scikit-learn.
2.	Train the SVM model on the training data.
3.	Evaluate model performance on the testing data using appropriate metrics (e.g., accuracy, precision, recall, F1-score).

Task 5: Visualization of SVM Results
1.	Visualize classification results on the testing data.

Task 6: Parameter Tuning and Optimization
1.	Experiment with different SVM hyperparameters (e.g., kernel type, regularization parameter) to optimize performance.

Task 7: Comparison and Analysis
1.	Compare SVM performance with various kernels (e.g., linear, polynomial, radial basis function).
2.	Analyze SVM strengths and weaknesses for the dataset based on EDA and visualization results.
3.	Discuss practical implications of SVM in real-world classification tasks.


```

## Answers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


dataset = pd.read_csv("Pharma_industry.csv")

df = dataset.copy()
print(df)

#Understanding Data
print("\n<---------INFO------->\n")
print(df.info())

print("\n<--------DWSCRIBE------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL VARIABLES------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES------->\n")
print(df.isnull().sum())

In [ ]:
# Univariate EDA
import seaborn as sns


numerical_cols = [col for col in df.columns.tolist() if not col =='Drug Response']

for col in numerical_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    plt.title(f"Histogram for {col}")
    sns.histplot(df[col],kde=True)

    plt.subplot(1,2,2)
    plt.title(f"Boxplot for {col}")
    sns.boxplot(df[col])

    plt.show()


In [ ]:
# Bivariate data analysis

for i,x_col in enumerate(numerical_cols):
    for y_col in numerical_cols[i+1:]:
        plt.figure(figsize=(18,9))
        plt.title(f"Scatterpolot between {x_col} and {y_col} ")
        plt.xlabel(f"{x_col}")
        plt.ylabel(f"{y_col}")
        plt.scatter(x=x_col, y=y_col,data=df)
        plt.show()
        

### Information from these graphs.
```
Univariate Analysis;
i.) Most of the distribution plot are continuous means they can have value within that interval.
ii) In some case there si some discrete value that shows the discontinuity of plot.
iii) Fron the boxplots we can hacve the idea about outliers.
iv) In every plot there are some outliers which are outside from both lower and upper whiskers as shown in dot points.

Bivariate Analysis between numerical variables:
i) Most of the scatter plots shows that there is no much relationship between the independent variables.
ii) So multicolinearity problem is low in this case.

```

### Data Cleaning
#### Points To Be Noted:
```
1.) There is no empty value as we have seen in above using isnull() function.
2.) But form the EDA we can clearly see from the boxplot there are some outliers in our dataset.

Hence, We have to treat them.
```

In [ ]:
# Finding the outliers

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    
    # Clippling the Outliers
    df[col] = df[col].clip(lower=lower_limit, upper = upper_limit)

print(df.shape)
    

## Why clipping not removing?
```
Ans: Here, there are many outliers in dtaset, so removing them could reduce the dataset. Almost 60% of i could lost if i removed all the Ourliers.
- If there would have any relationship between the variable then cliping them could dirupt the relation ship and the output also.
- But,here as you can see from the scatterplot  there is no much relationship between the variables so , we can clip them.

```

In [ ]:
# Checking whethere there is any outliers after cliping
for col in numerical_cols:
    plt.figure(figsize=(20,9))
    sns.boxplot(df[col])
    plt.show()
    

## Data Partition And Model Building

In [ ]:
X = df.iloc[:, :-1] # Independent variables
y = df.iloc[:, -1] # target_variable/ Dependent variable

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Splitting data into train and test 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Defining model
svm_model = SVC()



parameters = {
    'C': [0.1, 1, 10, 100],          
    'gamma': [1, 0.1, 0.01, 0.001], 
    'kernel': ['rbf', 'linear']     
}

svm_grid_model = GridSearchCV(svm_model,parameters, cv=5, refit=True)
svm_grid_model.fit(X_train,y_train)


In [ ]:
print("Best parameters:", svm_grid_model.best_params_)
print("Best CV accuracy:", svm_grid_model.best_score_)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
y_pred = svm_grid_model.predict(X_test)

print("Test accuracy is:",accuracy_score(y_pred,y_test))
print("\n<--------classification report is given by---------->\n")
print(classification_report(y_pred, y_test))

In [ ]:
# Comparing SVM performance with various kernels (e.g., linear, polynomial, radial basis function).
svm_linear = SVC(kernel='linear', C=1.0)
svm_poly = SVC(kernel='poly', degree=3, C=1.0)
svm_rbf = SVC(kernel='rbf', gamma=0.5, C=1.0)

# Train and evaluate SVM with linear kernel
svm_linear.fit(X_train, y_train)
y_pred_linear = svm_linear.predict(X_test)
accuracy_linear = accuracy_score(y_test, y_pred_linear)

# Train and evaluate SVM with polynomial kernel
svm_poly.fit(X_train, y_train)
y_pred_poly = svm_poly.predict(X_test)
accuracy_poly = accuracy_score(y_test, y_pred_poly)

# Train and evaluate SVM with RBF kernel
svm_rbf.fit(X_train, y_train)
y_pred_rbf = svm_rbf.predict(X_test)
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)


# Display results
print(f"Linear SVM Accuracy: {accuracy_linear:.2f}")
print(f"Polynomial SVM Accuracy: {accuracy_poly:.2f}")
print(f"RBF SVM Accuracy: {accuracy_rbf:.2f}")

In [ ]:
# PCA 
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import Delaunay

pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X)

# Trainning SVM Models on 3D Transformed Data
svm_linear_3d = SVC(kernel='linear', C=1.0)
svm_linear_3d.fit(X_pca_3d, y)

svm_poly_3d = SVC(kernel='poly', degree=3, C=1.0)
svm_poly_3d.fit(X_pca_3d, y)

svm_rbf_3d = SVC(kernel='rbf', gamma=0.5, C=1.0)
svm_rbf_3d.fit(X_pca_3d, y)

#  Function to Plot 3D Decision Boundaries
def plot_3d_decision_boundary_fixed(model, X, y, title):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Scatter plot of data points
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=y, cmap=plt.cm.Paired, edgecolor='k')

    # Create a triangulation of the feature space
    tri = Delaunay(X[:, :2])

    # Predict values for the 3D surface
    Z = model.decision_function(X)

    # Create surface plot
    ax.plot_trisurf(X[:, 0], X[:, 1], Z, triangles=tri.simplices, cmap='coolwarm', alpha=0.3)

    ax.set_xlabel('Principal Component 1')
    ax.set_ylabel('Principal Component 2')
    ax.set_zlabel('Decision Boundary')
    ax.set_title(title)

    plt.show()

# Plotting Decision Boundaries for Each Model
%matplotlib qt
plot_3d_decision_boundary_fixed(svm_linear_3d, X_pca_3d, y, "3D Decision Boundary - Linear SVM")

%matplotlib qt
plot_3d_decision_boundary_fixed(svm_poly_3d, X_pca_3d, y, "3D Decision Boundary - Polynomial SVM")

%matplotlib qt
plot_3d_decision_boundary_fixed(svm_rbf_3d, X_pca_3d, y, "3D Decision Boundary - RBF SVM")



## Practical Implementations:
```
Ans: Real world applications of SVM  are:
1.) Text and document classification - Like spam detection, sentiment anslysis.
2.) Image classification - face recognition, object detection, Handwritten digit recognition
3.) Biometrics/ Health Care - Deseases predicatio models, gene expression data analysis. 
```
###                              <-------------------------------END------------------------------------->